In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import ephem
from datetime import datetime, timezone

PROJECT_ROOT = Path.home() / "wspr-gannon"
WSPR_DATA_DIR  = PROJECT_ROOT / "data" / "wspr"
SW_DATA_DIR    = PROJECT_ROOT / "data" / "spaceweather"

print("Geometry notebook ready.")

In [ ]:
def solar_zenith_angle(lat: float, lon: float, dt: datetime) -> float:
    """
    Compute solar zenith angle (degrees) at a given location and time.
    0° = sun directly overhead, 90° = horizon, >90° = night.
    """
    obs = ephem.Observer()
    obs.lat = str(lat)
    obs.lon = str(lon)
    obs.date = dt.strftime("%Y/%m/%d %H:%M:%S")
    obs.pressure = 0  # no refraction correction
    
    sun = ephem.Sun(obs)
    # ephem gives altitude above horizon; zenith = 90 - altitude
    altitude_deg = float(sun.alt) * 180.0 / np.pi
    return 90.0 - altitude_deg

def is_daylight(lat: float, lon: float, dt: datetime) -> bool:
    """True if sun is above horizon at this location and time."""
    return solar_zenith_angle(lat, lon, dt) < 90.0

def path_midpoint(tx_lat: float, tx_lon: float, 
                  rx_lat: float, rx_lon: float) -> tuple:
    """
    Compute great circle midpoint between two points.
    Returns (lat, lon) in degrees.
    Simple mean works well for paths < 5000km; 
    for longer paths we use the spherical midpoint.
    """
    # Convert to radians
    lat1, lon1 = np.radians(tx_lat), np.radians(tx_lon)
    lat2, lon2 = np.radians(rx_lat), np.radians(rx_lon)
    
    # Cartesian midpoint on unit sphere
    x1, y1, z1 = np.cos(lat1)*np.cos(lon1), np.cos(lat1)*np.sin(lon1), np.sin(lat1)
    x2, y2, z2 = np.cos(lat2)*np.cos(lon2), np.cos(lat2)*np.sin(lon2), np.sin(lat2)
    
    xm = (x1 + x2) / 2
    ym = (y1 + y2) / 2
    zm = (z1 + z2) / 2
    
    mid_lat = np.degrees(np.arctan2(zm, np.sqrt(xm**2 + ym**2)))
    mid_lon = np.degrees(np.arctan2(ym, xm))
    
    return mid_lat, mid_lon

def magnetic_latitude(geo_lat: float, geo_lon: float) -> float:
    """
    Approximate magnetic latitude from geographic coordinates.
    Uses the centered dipole approximation with 2023 pole position.
    North magnetic pole ~86.4°N, 162.7°W as of ~2023.
    Good enough for propagation analysis; not a substitute for IGRF.
    """
    pole_lat = np.radians(86.4)
    pole_lon = np.radians(-162.7)
    lat_r    = np.radians(geo_lat)
    lon_r    = np.radians(geo_lon)
    
    sin_mlat = (np.sin(pole_lat) * np.sin(lat_r) +
                np.cos(pole_lat) * np.cos(lat_r) * np.cos(lon_r - pole_lon))
    return np.degrees(np.arcsin(sin_mlat))

# Quick sanity checks
dt_noon = datetime(2023, 6, 15, 12, 0, 0)
dt_midnight = datetime(2023, 6, 15, 0, 0, 0)

print(f"Solar zenith at London noon UTC:     {solar_zenith_angle(51.5, -0.1, dt_noon):.1f}°")
print(f"Solar zenith at London midnight UTC: {solar_zenith_angle(51.5, -0.1, dt_midnight):.1f}°")
print(f"Daylight at London noon:     {is_daylight(51.5, -0.1, dt_noon)}")
print(f"Daylight at London midnight: {is_daylight(51.5, -0.1, dt_midnight)}")

mid = path_midpoint(40.0, -75.0, 51.5, -0.1)  # NYC to London
print(f"\nPath midpoint NYC→London: {mid[0]:.1f}°N, {mid[1]:.1f}°E")

mlat = magnetic_latitude(51.5, -0.1)
print(f"Magnetic latitude of London: {mlat:.1f}°")

In [ ]:
def add_geometry(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add geometric columns to a WSPR DataFrame.
    Operates row-wise — expects time, tx_lat, tx_lon, rx_lat, rx_lon columns.
    """
    print(f"Computing geometry for {len(df):,} rows...")
    
    # Path midpoint
    mid = np.vectorize(path_midpoint)(
        df["tx_lat"].values, df["tx_lon"].values,
        df["rx_lat"].values, df["rx_lon"].values
    )
    df["mid_lat"] = mid[0]
    df["mid_lon"] = mid[1]
    
    # Magnetic latitude of midpoint
    df["mid_mlat"] = np.vectorize(magnetic_latitude)(
        df["mid_lat"].values, df["mid_lon"].values
    )
    
    # Solar zenith angle at path midpoint
    # Convert time to datetime for ephem
    times = df["time"].dt.to_pydatetime()
    
    df["mid_sza"] = [
        solar_zenith_angle(row["mid_lat"], row["mid_lon"], t)
        for row, t in zip(df[["mid_lat","mid_lon"]].to_dict("records"), times)
    ]
    
    # Daylight flags for tx and rx separately
    df["tx_daylight"] = [
        is_daylight(row["tx_lat"], row["tx_lon"], t)
        for row, t in zip(df[["tx_lat","tx_lon"]].to_dict("records"), times)
    ]
    df["rx_daylight"] = [
        is_daylight(row["rx_lat"], row["rx_lon"], t)
        for row, t in zip(df[["rx_lat","rx_lon"]].to_dict("records"), times)
    ]
    
    # Path type: both day, both night, or mixed (gray line)
    conditions = [
        df["tx_daylight"] & df["rx_daylight"],
        ~df["tx_daylight"] & ~df["rx_daylight"],
    ]
    df["path_type"] = np.select(conditions, ["day", "night"], default="mixed")
    
    return df

# Test on one day of data
print("Loading test day...")
test_df = pd.read_parquet(WSPR_DATA_DIR / "wspr_2023-06-15_20m.parquet")
print(f"Loaded {len(test_df):,} rows")

print("add_geometry() defined — skipping 2023 test, using Gannon data.")

In [ ]:
def add_geometry_fast(df: pd.DataFrame) -> pd.DataFrame:
    """
    Vectorized geometry computation exploiting WSPR's 2-minute cadence.
    Most spots share timestamps — compute per unique (time, grid) combination.
    """
    print(f"Computing geometry for {len(df):,} rows...")
    
    # Path midpoint — fully vectorized with numpy
    lat1 = np.radians(df["tx_lat"].values)
    lon1 = np.radians(df["tx_lon"].values)
    lat2 = np.radians(df["rx_lat"].values)
    lon2 = np.radians(df["rx_lon"].values)
    
    x = np.cos(lat1)*np.cos(lon1) + np.cos(lat2)*np.cos(lon2)
    y = np.cos(lat1)*np.sin(lon1) + np.cos(lat2)*np.sin(lon2)
    z = np.sin(lat1) + np.sin(lat2)
    
    df["mid_lat"] = np.degrees(np.arctan2(z, np.sqrt(x**2 + y**2)))
    df["mid_lon"] = np.degrees(np.arctan2(y, x))
    
    # Magnetic latitude — fully vectorized
    pole_lat = np.radians(86.4)
    pole_lon = np.radians(-162.7)
    mlat_r = np.radians(df["mid_lat"].values)
    mlon_r = np.radians(df["mid_lon"].values)
    sin_mlat = (np.sin(pole_lat) * np.sin(mlat_r) +
                np.cos(pole_lat) * np.cos(mlat_r) * np.cos(mlon_r - pole_lon))
    df["mid_mlat"] = np.degrees(np.arcsin(np.clip(sin_mlat, -1, 1)))

    # Solar zenith — compute on unique (time, mid_lat_r, mid_lon_r) combinations
    # Round midpoint to 1 degree to increase cache hits
    df["_mid_lat_r"] = df["mid_lat"].round(0)
    df["_mid_lon_r"] = df["mid_lon"].round(0)
    
    unique_combos = df[["time", "_mid_lat_r", "_mid_lon_r"]].drop_duplicates()
    print(f"  Unique (time, location) combos: {len(unique_combos):,} vs {len(df):,} rows")
    
    def compute_sza_row(row):
        obs = ephem.Observer()
        obs.lat  = str(row["_mid_lat_r"])
        obs.lon  = str(row["_mid_lon_r"])
        obs.date = row["time"].strftime("%Y/%m/%d %H:%M:%S")
        obs.pressure = 0
        sun = ephem.Sun(obs)
        alt_deg = float(sun.alt) * 180.0 / np.pi
        return 90.0 - alt_deg
    
    unique_combos["mid_sza"] = unique_combos.apply(compute_sza_row, axis=1)
    
    df = df.merge(unique_combos[["time","_mid_lat_r","_mid_lon_r","mid_sza"]],
                  on=["time","_mid_lat_r","_mid_lon_r"], how="left")
    df = df.drop(columns=["_mid_lat_r","_mid_lon_r"])
    
    # Tx/Rx daylight from SZA at their locations
    # Round to 1 degree for same caching trick
    df["_tx_lat_r"] = df["tx_lat"].round(0)
    df["_tx_lon_r"] = df["tx_lon"].round(0)
    df["_rx_lat_r"] = df["rx_lat"].round(0)
    df["_rx_lon_r"] = df["rx_lon"].round(0)
    
    for prefix in [("tx", "_tx_lat_r", "_tx_lon_r"), 
                   ("rx", "_rx_lat_r", "_rx_lon_r")]:
        name, lat_col, lon_col = prefix
        unique_end = df[["time", lat_col, lon_col]].drop_duplicates()
        
        def sza_end(row, lc=lat_col, vc=lon_col):
            obs = ephem.Observer()
            obs.lat  = str(row[lc])
            obs.lon  = str(row[vc])
            obs.date = row["time"].strftime("%Y/%m/%d %H:%M:%S")
            obs.pressure = 0
            sun = ephem.Sun(obs)
            return 90.0 - float(sun.alt) * 180.0 / np.pi
        
        unique_end[f"{name}_sza"] = unique_end.apply(sza_end, axis=1)
        df = df.merge(unique_end[["time", lat_col, lon_col, f"{name}_sza"]],
                      on=["time", lat_col, lon_col], how="left")
        df = df.drop(columns=[lat_col, lon_col])
    
    df["tx_daylight"] = df["tx_sza"] < 90.0
    df["rx_daylight"] = df["rx_sza"] < 90.0
    
    conditions = [
        df["tx_daylight"] & df["rx_daylight"],
        ~df["tx_daylight"] & ~df["rx_daylight"],
    ]
    df["path_type"] = np.select(conditions, ["day", "night"], default="mixed")
    
    return df

print("add_geometry_fast() defined.")

In [ ]:
from scipy.interpolate import RegularGridInterpolator

def build_sza_interpolator(date_str: str) -> RegularGridInterpolator:
    """
    Build a 3D interpolator for solar zenith angle over a full day.
    Grid: 48 time steps × 19 lat steps × 37 lon steps = 33,744 ephem calls
    vs potentially millions of row-level calls.
    """
    # Grid definition
    # 30-minute time steps covering full day
    times_hm = np.arange(0, 24*60, 30)       # minutes since midnight
    lats      = np.arange(-5, 80, 5)          # 5° steps, our data range
    lons      = np.arange(-135, 45, 5)        # 5° steps, NA+Europe

    sza_grid = np.zeros((len(times_hm), len(lats), len(lons)))

    year, month, day = [int(x) for x in date_str.split("-")]

    for i, tm in enumerate(times_hm):
        h, m = int(tm // 60), int(tm % 60)
        dt_str = f"{year}/{month:02d}/{day:02d} {h:02d}:{m:02d}:00"
        for j, lat in enumerate(lats):
            for k, lon in enumerate(lons):
                obs = ephem.Observer()
                obs.lat      = str(lat)
                obs.lon      = str(lon)
                obs.date     = dt_str
                obs.pressure = 0
                sun = ephem.Sun(obs)
                sza_grid[i, j, k] = 90.0 - float(sun.alt) * 180.0 / np.pi

    interp = RegularGridInterpolator(
        (times_hm, lats, lons),
        sza_grid,
        method="linear",
        bounds_error=False,
        fill_value=None
    )
    return interp, times_hm, lats, lons

def add_geometry_interpolated(df: pd.DataFrame, date_str: str) -> pd.DataFrame:
    """
    Fast geometry using pre-computed SZA interpolation grid.
    """
    print(f"Building SZA interpolator for {date_str}...", end=" ")
    interp, _, _, _ = build_sza_interpolator(date_str)
    print("done.")

    print(f"Computing geometry for {len(df):,} rows...")

    # Path midpoint — vectorized
    lat1 = np.radians(df["tx_lat"].values)
    lon1 = np.radians(df["tx_lon"].values)
    lat2 = np.radians(df["rx_lat"].values)
    lon2 = np.radians(df["rx_lon"].values)

    x = np.cos(lat1)*np.cos(lon1) + np.cos(lat2)*np.cos(lon2)
    y = np.cos(lat1)*np.sin(lon1) + np.cos(lat2)*np.sin(lon2)
    z = np.sin(lat1) + np.sin(lat2)

    df["mid_lat"] = np.degrees(np.arctan2(z, np.sqrt(x**2 + y**2)))
    df["mid_lon"] = np.degrees(np.arctan2(y, x))

    # Magnetic latitude — vectorized
    pole_lat = np.radians(86.4)
    pole_lon = np.radians(-162.7)
    mlat_r   = np.radians(df["mid_lat"].values)
    mlon_r   = np.radians(df["mid_lon"].values)
    sin_mlat = (np.sin(pole_lat) * np.sin(mlat_r) +
                np.cos(pole_lat) * np.cos(mlat_r) * np.cos(mlon_r - pole_lon))
    df["mid_mlat"] = np.degrees(np.arcsin(np.clip(sin_mlat, -1, 1)))

    # Minutes since midnight for interpolation
    mins = (df["time"].dt.hour * 60 + df["time"].dt.minute).values

    # SZA at midpoint, tx, rx — all vectorized via interpolator
    df["mid_sza"] = interp((mins, df["mid_lat"].values, df["mid_lon"].values))
    df["tx_sza"]  = interp((mins, df["tx_lat"].values,  df["tx_lon"].values))
    df["rx_sza"]  = interp((mins, df["rx_lat"].values,  df["rx_lon"].values))

    df["tx_daylight"] = df["tx_sza"] < 90.0
    df["rx_daylight"] = df["rx_sza"] < 90.0

    conditions = [
        df["tx_daylight"] & df["rx_daylight"],
        ~df["tx_daylight"] & ~df["rx_daylight"],
    ]
    df["path_type"] = np.select(conditions, ["day", "night"], default="mixed")

    return df

print("build_sza_interpolator() and add_geometry_interpolated() defined.")

In [ ]:
from datetime import date, timedelta

# Gannon storm dates
gannon_dates = [date(2024, 5, d) for d in range(7, 15)]
BANDS = [10, 20, 40]

print(f"Gannon schedule: {len(gannon_dates)} days")

In [ ]:
def process_all(schedule: list, bands_m: list) -> None:
    """
    Process all WSPR files — add geometry and save enriched parquets.
    Builds SZA interpolator once per date, reuses across bands.
    """
    total = len(schedule) * len(bands_m)
    completed = 0

    for d in schedule:
        date_str = d.strftime("%Y-%m-%d")

        needs_processing = any(
            not (WSPR_DATA_DIR / f"wspr_{date_str}_{b}m_geo.parquet").exists()
            for b in bands_m
        )

        if not needs_processing:
            completed += len(bands_m)
            continue

        interp, _, _, _ = build_sza_interpolator(date_str)

        for band_m in bands_m:
            src = WSPR_DATA_DIR / f"wspr_{date_str}_{band_m}m.parquet"
            out = WSPR_DATA_DIR / f"wspr_{date_str}_{band_m}m_geo.parquet"

            if out.exists():
                completed += 1
                continue
            if not src.exists():
                completed += 1
                continue

            df = pd.read_parquet(src)

            lat1 = np.radians(df["tx_lat"].values)
            lon1 = np.radians(df["tx_lon"].values)
            lat2 = np.radians(df["rx_lat"].values)
            lon2 = np.radians(df["rx_lon"].values)
            x = np.cos(lat1)*np.cos(lon1) + np.cos(lat2)*np.cos(lon2)
            y = np.cos(lat1)*np.sin(lon1) + np.cos(lat2)*np.sin(lon2)
            z = np.sin(lat1) + np.sin(lat2)
            df["mid_lat"] = np.degrees(np.arctan2(z, np.sqrt(x**2 + y**2)))
            df["mid_lon"] = np.degrees(np.arctan2(y, x))

            pole_lat = np.radians(86.4)
            pole_lon = np.radians(-162.7)
            mlat_r   = np.radians(df["mid_lat"].values)
            mlon_r   = np.radians(df["mid_lon"].values)
            sin_mlat = (np.sin(pole_lat) * np.sin(mlat_r) +
                        np.cos(pole_lat) * np.cos(mlat_r) *
                        np.cos(mlon_r - pole_lon))
            df["mid_mlat"] = np.degrees(np.arcsin(np.clip(sin_mlat, -1, 1)))

            mins = (df["time"].dt.hour * 60 + df["time"].dt.minute).values
            df["mid_sza"] = interp((mins, df["mid_lat"].values, df["mid_lon"].values))
            df["tx_sza"]  = interp((mins, df["tx_lat"].values,  df["tx_lon"].values))
            df["rx_sza"]  = interp((mins, df["rx_lat"].values,  df["rx_lon"].values))

            df["tx_daylight"] = df["tx_sza"] < 90.0
            df["rx_daylight"] = df["rx_sza"] < 90.0
            conditions = [
                df["tx_daylight"] & df["rx_daylight"],
                ~df["tx_daylight"] & ~df["rx_daylight"],
            ]
            df["path_type"] = np.select(conditions, ["day","night"], default="mixed")

            df.to_parquet(out, index=False)
            completed += 1

        pct = 100 * completed / total
        print(f"[{pct:.0f}%] {date_str} done")

# Dry run
print("Checking what needs processing...")
needs = sum(
    1 for d in gannon_dates for b in [10,20,40]
    if not (WSPR_DATA_DIR / f"wspr_{d.strftime('%Y-%m-%d')}_{b}m_geo.parquet").exists()
)
print(f"Files to process: {needs}/24")
print(f"Estimated time: {needs * 0.9 / 60:.1f} minutes")

In [ ]:
process_all(gannon_dates, BANDS)
print("All geometry processing complete.")

In [ ]:
# Spot check a Gannon date
sample = pd.read_parquet(WSPR_DATA_DIR / "wspr_2024-05-07_20m_geo.parquet")
print(f"Rows: {len(sample):,}")
print(f"Columns: {list(sample.columns)}")
print(f"\nPath types: {sample['path_type'].value_counts().to_dict()}")
print(f"Mid SZA range: {sample['mid_sza'].min():.1f}° to {sample['mid_sza'].max():.1f}°")
print(f"Mid mlat range: {sample['mid_mlat'].min():.1f}° to {sample['mid_mlat'].max():.1f}°")
display(sample.head(3))

In [ ]:
import os
all_files = [f for f in os.listdir(WSPR_DATA_DIR) if '2024' in f]
print(f"2024 files found: {len(all_files)}")
for f in sorted(all_files):
    print(f"  {f}")

In [ ]:
print(f"WSPR_DATA_DIR: {WSPR_DATA_DIR}")
print(f"Files in dir: {len(os.listdir(WSPR_DATA_DIR))}")

In [ ]:
import os
wspr_orig = Path.home() / "wspr-propagation" / "data" / "wspr"
files_2024 = [f for f in os.listdir(wspr_orig) if '2024' in f]
print(f"2024 files in wspr-propagation: {len(files_2024)}")
for f in sorted(files_2024)[:6]:
    print(f"  {f}")

In [ ]:
import shutil

wspr_orig = Path.home() / "wspr-propagation" / "data" / "wspr"
files_2024 = [f for f in os.listdir(wspr_orig) if '2024' in f]

for f in files_2024:
    src = wspr_orig / f
    dst = WSPR_DATA_DIR / f
    shutil.copy2(src, dst)
    
print(f"Copied {len(files_2024)} files to {WSPR_DATA_DIR}")

# Verify
files_check = [f for f in os.listdir(WSPR_DATA_DIR) if '2024' in f]
print(f"2024 files now in wspr-gannon: {len(files_check)}")